In [3]:
import os
import zipfile
import numpy as np
import tensorflow as tf

from tensorflow.keras.layers import (
    Input,
    Conv1D,
    DepthwiseConv1D,
    MaxPooling1D,
    GlobalAveragePooling1D,
    Dense
)

from tensorflow.keras.models import Model
from tensorflow.keras.utils import to_categorical

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)


# ============================================================
# 1. Seed
# ============================================================

SEED = 42

np.random.seed(SEED)
tf.random.set_seed(SEED)


# ============================================================
# 2. Path
# ============================================================

DATA_PATH = "/content/drive/MyDrive/Colab Notebooks/UCI HAR Dataset"


# ============================================================
# 3. Download
# ============================================================

if not os.path.exists(DATA_PATH):

    url = (
        "https://archive.ics.uci.edu/ml/machine-learning-databases/"
        "00240/UCI%20HAR%20Dataset.zip"
    )

    zip_path = tf.keras.utils.get_file(
        "UCI_HAR.zip",
        origin=url,
        cache_dir="/content",
        cache_subdir=""
    )

    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(
            "/content/drive/MyDrive/Colab Notebooks"
        )


# ============================================================
# 4. Signals
# ============================================================

signals = [
    "body_acc_x",
    "body_acc_y",
    "body_acc_z",
    "body_gyro_x",
    "body_gyro_y",
    "body_gyro_z",
    "total_acc_x",
    "total_acc_y",
    "total_acc_z"
]


# ============================================================
# 5. Load Dataset
# ============================================================

def load_data(split):

    X = []

    for signal in signals:

        file_path = (
            f"{DATA_PATH}/{split}/"
            f"Inertial Signals/{signal}_{split}.txt"
        )

        X.append(
            np.loadtxt(file_path)
        )

    X = np.stack(
        X,
        axis=-1
    )

    y = np.loadtxt(
        f"{DATA_PATH}/{split}/"
        f"y_{split}.txt"
    ).astype(int)

    y = y - 1

    return X, y


X_train, y_train = load_data("train")
X_test, y_test = load_data("test")


# ============================================================
# 6. Normalization
# ============================================================

mean = X_train.mean(
    axis=(0, 1),
    keepdims=True
)

std = X_train.std(
    axis=(0, 1),
    keepdims=True
)

X_train = (
    X_train - mean
) / (
    std + 1e-8
)

X_test = (
    X_test - mean
) / (
    std + 1e-8
)


# ============================================================
# 7. One-hot
# ============================================================

y_train_onehot = to_categorical(
    y_train,
    6
)

y_test_onehot = to_categorical(
    y_test,
    6
)


# ============================================================
# 8. Model
# ============================================================

inputs = Input(
    shape=(128, 9)
)


# ------------------------------------------------------------
# Block 1
# ------------------------------------------------------------

x = Conv1D(
    32,
    3,
    padding="same",
    activation="relu"
)(inputs)


x = DepthwiseConv1D(
    3,
    padding="same",
    activation="relu"
)(x)


# Pointwise convolution

x = Conv1D(
    64,
    1,
    padding="same",
    activation="relu"
)(x)


x = MaxPooling1D(
    2
)(x)


# ------------------------------------------------------------
# Block 2
# ------------------------------------------------------------

x = DepthwiseConv1D(
    3,
    padding="same",
    activation="relu"
)(x)


x = Conv1D(
    128,
    1,
    padding="same",
    activation="relu"
)(x)


x = MaxPooling1D(
    2
)(x)


# ------------------------------------------------------------
# Block 3
# ------------------------------------------------------------

x = DepthwiseConv1D(
    3,
    padding="same",
    activation="relu"
)(x)


x = Conv1D(
    128,
    1,
    padding="same",
    activation="relu"
)(x)


# ============================================================
# 9. Classifier
# ============================================================

x = GlobalAveragePooling1D()(x)


x = Dense(
    128,
    activation="relu"
)(x)


outputs = Dense(
    6,
    activation="softmax"
)(x)


model = Model(
    inputs,
    outputs
)


# ============================================================
# 10. Compile
# ============================================================

model.compile(
    loss="categorical_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)


model.summary()


# ============================================================
# 11. Train
# ============================================================

history = model.fit(
    X_train,
    y_train_onehot,
    epochs=20,
    batch_size=64,
    validation_split=0.2,
    shuffle=True,
    verbose=1
)


# ============================================================
# 12. Prediction
# ============================================================

y_prob = model.predict(
    X_test,
    verbose=0
)

y_pred = np.argmax(
    y_prob,
    axis=1
)


# ============================================================
# 13. Evaluation
# ============================================================

accuracy = accuracy_score(
    y_test,
    y_pred
)

precision = precision_score(
    y_test,
    y_pred,
    average="macro",
    zero_division=0
)

recall = recall_score(
    y_test,
    y_pred,
    average="macro",
    zero_division=0
)

f1 = f1_score(
    y_test,
    y_pred,
    average="macro",
    zero_division=0
)


print("\n==============================")
print("DEPTHWISE TEST RESULTS")
print("==============================")

print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1-score  : {f1:.4f}")


# ============================================================
# 14. Classification Report
# ============================================================

class_names = [
    "WALKING",
    "WALKING_UPSTAIRS",
    "WALKING_DOWNSTAIRS",
    "SITTING",
    "STANDING",
    "LAYING"
]


print(
    classification_report(
        y_test,
        y_pred,
        target_names=class_names,
        digits=4,
        zero_division=0
    )
)


# ============================================================
# 15. Confusion Matrix
# ============================================================

cm = confusion_matrix(
    y_test,
    y_pred
)

print("\nConfusion Matrix")

print(cm)

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 128, 9)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_8 (Conv1D)               │ (None, 128, 32)        │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ depthwise_conv1d_6              │ (None, 128, 32)        │           128 │
│ (DepthwiseConv1D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_9 (Conv1D)               │ (None, 128, 64)        │         2,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_4 (MaxPooling1D)  │ (None, 64, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ depthwise_conv1d_7              │ (None, 64, 64)         │           256 │
│ (DepthwiseConv1D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_10 (Conv1D)              │ (None, 64, 128)        │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_5 (MaxPooling1D)  │ (None, 32, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ depthwise_conv1d_8              │ (None, 32, 128)        │           512 │
│ (DepthwiseConv1D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_11 (Conv1D)              │ (None, 32, 128)        │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_2      │ (None, 128)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 6)              │           774 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 46,022 (179.77 KB)

 Trainable params: 46,022 (179.77 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
92/92 ━━━━━━━━━━━━━━━━━━━━ 12s 51ms/step - accuracy: 0.3710 - loss: 1.5093 - val_accuracy: 0.6370 - val_loss: 0.8371
Epoch 2/20
92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.7283 - loss: 0.5737 - val_accuracy: 0.7723 - val_loss: 0.5335
Epoch 3/20
92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.8216 - loss: 0.3825 - val_accuracy: 0.8559 - val_loss: 0.3840
Epoch 4/20
92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.8823 - loss: 0.2823 - val_accuracy: 0.8933 - val_loss: 0.3289
Epoch 5/20
92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9100 - loss: 0.2176 - val_accuracy: 0.8953 - val_loss: 0.2901
Epoch 6/20
92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9294 - loss: 0.1735 - val_accuracy: 0.9096 - val_loss: 0.2602
Epoch 7/20
92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.9432 - loss: 0.1439 - val_accuracy: 0.9252 - val_loss: 0.2435
Epoch 8/20
92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.9480 - loss: 0.1287 - val_accuracy: 0.9307 - val_los